In [ ]:
# --- 0) Path/Import ---
import os, sys, json, math, glob
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pickle
from types import SimpleNamespace
from hydra.utils import instantiate

# Add paths relative to the current notebook location
# Assumes notebook is in: TimeTok_Repo/00_Hierarchical_Tokenizer/notebook/
sys.path.append(os.path.abspath("../../../"))
sys.path.append(os.path.abspath("../../../TSgen"))   # TSgen package root

from TSgen.utils.misc import detect_bf16_support, get_bf16_context, get_generator
from TSgen.utils.demo import imgs_from_urls, denormalize, batch_to_pil
import torchvision.transforms.functional as TF

enable_bf16 = detect_bf16_support()

from TSgen.data.UTSD_data_factory import utsd_data_provider
from TSgen.data.UCR_data_factory import ucr_data_provider
from TSgen.tsgen_wrapper import TSgen  # For _target_ verification/instantiation

# --- 1) Setup: Dataset/Checkpoint Root ---
DATASET   = "ECG5000"   # Replace if necessary
UCR_ROOT  = os.path.abspath("../../../TSgen/data/UCRConverted")

# --- 2) Prepare data loader (val) ---
ns = SimpleNamespace(dataset=DATASET, data_root=UCR_ROOT, root_path=UCR_ROOT,
                     num_workers=0, seq_len=None, seed=42)
train_ls, val_ls, _, param = ucr_data_provider(ns)
train_loader = train_ls[0]
val_loader = val_ls[0]

# # # UTSD
# args = SimpleNamespace(
#     ucr_root=os.path.abspath("../../../TSgen/data/UTSD-full-npy"),
#     num_workers=8,
#     batch_size=256,
#     subset_ratio=0.0002
# )

# train_loader, val_loader = utsd_data_provider(args, "train"), utsd_data_provider(args, "val")

In [2]:
import numpy as np
import torch
from typing import List, Tuple

# =========================
# DFA (1st order) Utils
# =========================
def default_dfa_scales_frac(L: int, s_min_frac: float = 0.02, s_max_frac: float = 0.25, num_scales: int = 12) -> List[int]:
    s_min = max(4, int(round(L * s_min_frac)))
    s_max = max(s_min + 1, int(round(L * s_max_frac)))
    raw = np.unique(
        np.clip(
            np.round(np.logspace(np.log10(s_min), np.log10(s_max), num_scales)),
            s_min, s_max
        ).astype(int)
    )
    # Ensure minimum number of segments (>=4) & L/4 upper bound
    scales = [int(s) for s in raw if (s >= s_min) and (s <= L//4) and (L // s >= 4)]
    return scales

def dfa_alpha_batch_torch(
    x_bl: torch.Tensor,  
    scales: List[int],
    order: int = 1
) -> torch.Tensor:
    """
    Batch DFA (1st order only). Returns: alphas [B]
    """
    assert x_bl.ndim == 2, "x_bl must be [B, L]"
    assert order == 1, "Batch DFA supports only 1st order with minimal modifications."
    B, L = x_bl.shape
    device = x_bl.device

    # 1) Remove mean and compute cumulative sum
    x = x_bl - x_bl.mean(dim=1, keepdim=True)
    y = torch.cumsum(x, dim=1)  # [B, L]

    Fs = []     # F(s) for each s [B]
    s_kept = [] # List of actually used s (satisfying conditions)

    for s in scales:
        if s < 4:
            continue
        Ns = L // s
        if Ns < 4:
            continue
        # --- forward windows: [B, Ns, s]
        fw = y.unfold(dimension=1, size=s, step=s)  # [B, Ns, s]
        # --- backward windows: Pad with equal number if leftovers exist
        if (L % s) != 0:
            # Same as Ns equal partitions from the back: reverse then unfold
            y_rev = torch.flip(y, dims=[1])
            bw = y_rev.unfold(dimension=1, size=s, step=s)[:, :Ns, :]  # [B, Ns, s]
            windows = torch.cat([fw, bw], dim=1)  # [B, 2*Ns, s]
        else:
            windows = fw  # [B, Ns, s]
        # --- Linear detrend (closed form) ---
        # t = [0,1,...,s-1]
        t = torch.arange(s, device=device, dtype=x.dtype).view(1, 1, s)  # [1,1,s]
        N = float(s)
        sum_t  = (s - 1) * s * 0.5                        # Σ t
        sum_t2 = (s - 1) * s * (2 * s - 1) / 6.0          # Σ t^2

        y_sum  = windows.sum(dim=2)                       # [B, M]
        ty_sum = (windows * t).sum(dim=2)                 # [B, M]

        denom = (N * sum_t2 - sum_t * sum_t)             # scalar
        a1 = (N * ty_sum - sum_t * y_sum) / denom        # slope [B, M]
        a0 = (y_sum - a1 * sum_t) / N                    # intercept [B, M]

        trend = a1.unsqueeze(-1) * t + a0.unsqueeze(-1)  # [B, M, s]
        resid = windows - trend                          # [B, M, s]
        rms = torch.sqrt((resid ** 2).mean(dim=2))       # [B, M]
        F_s = torch.sqrt((rms ** 2).mean(dim=1))       # [B],  F(n) = sqrt(mean(F(n,i)^2))
        # F_s = rms.mean(dim=1)                            # [B]
        Fs.append(F_s)
        s_kept.append(s)
    if len(Fs) < 2:
        # Cannot regress due to insufficient scales -> Return NaN (same as original policy)
        return torch.full((B,), float('nan'), device=device)
    # 2) log–log regression (alpha per sample)
    F_mat = torch.stack(Fs, dim=1)              # [B, S’]
    s_arr = torch.tensor(s_kept, dtype=x.dtype) # [S’] on CPU? move to device
    s_arr = s_arr.to(device)

    # mask: F>0
    mask = F_mat > 0
    xs = torch.log(s_arr).view(1, -1).expand(B, -1)      # [B, S’]
    ys = torch.log(torch.clamp(F_mat, min=1e-12))        # [B, S’]

    # Use only valid scales
    # (Calculate mean-covariance using mask to handle sample-specific masks simply)
    def masked_mean(t, m):
        w = m.float()
        return (t * w).sum(dim=1) / torch.clamp(w.sum(dim=1), min=1.0)

    mx = masked_mean(xs, mask)            # [B]
    my = masked_mean(ys, mask)            # [B]
    vx = masked_mean((xs - mx.unsqueeze(1))**2, mask)     # [B]
    cxy = masked_mean((xs - mx.unsqueeze(1))*(ys - my.unsqueeze(1)), mask)  # [B]

    # slope = cov/var
    alphas = cxy / torch.clamp(vx, min=1e-12)            # [B]
    return alphas
# =========================
# Gaussian smoothing (NumPy)
# =========================
def gaussian_smooth_1d(x: np.ndarray, sigma: float) -> np.ndarray:
    if sigma <= 0.0:
        return x
    L = x.shape[0]
    half = int(3 * sigma + 0.5)
    half = min(half, max(0, (L - 1) // 2))
    if half == 0:
        return x
    K = 2 * half + 1
    grid = np.arange(-half, half + 1, dtype=float)
    ker = np.exp(-0.5 * (grid / sigma) ** 2)
    ker = ker / ker.sum()
    x_pad = np.pad(x, (half, half), mode="reflect")
    y = np.convolve(x_pad, ker, mode="valid")
    return y

def window_candidates(L: int) -> np.ndarray:
    W_min = 1
    W_max = max(1, L//2)
    # Include both ends, 8 equal steps
    W_candidates = np.linspace(W_min, W_max, num=8, dtype=float)
    return W_candidates

def sigma_from_W(W: float) -> float:
    # 6*sigma + 1 ≈ W  →  sigma = (W-1)/6
    return max(0.0, (float(W) - 1.0) / 6.0)

def compute_alpha_edges_equalized(
    train_loader,
    max_samples: int = 100,
    num_levels: int = 8,
    dfa_order: int = 1,
    dfa_num_scales: int = 12,
    dfa_smin_frac: float = 0.02,
    dfa_smax_frac: float = 0.25,
    seed: int = 0,
) -> List[float]:
    """
    Returns: alpha_edges (length num_levels-1) — Quantile boundaries ensuring equal ratio (≈1/num_levels) for each level.
    Usage: np.searchsorted(alpha_edges, alpha, 'left') → bin idx(0..num_levels-1)
         levels are mapped inversely (e.g., [1,2,4,8,16,32,64,128]) because smaller α means more tokens.
    """
    assert dfa_order == 1, "Current batch DFA supports only 1st order."
    rng = np.random.default_rng(seed)
    alphas = []

    collected = 0
    for batch in train_loader:
        x = batch[0].float().cpu()  # [B,C,L]  (dfa can be on CPU)
        B, C, L = x.shape
        x_1c = x[:, 0, :].numpy()   # [B, L]  (Smoothing kept in NumPy)

        # Scales depend only on length L → generate once per batch
        scales = default_dfa_scales_frac(
            L=L, s_min_frac=dfa_smin_frac,
            s_max_frac=dfa_smax_frac, num_scales=dfa_num_scales
        )
        if len(scales) < 2:
            continue

        # Window candidates / sigma conversion (keep existing logic)
        W_cand = window_candidates(L)
        sigmas = [sigma_from_W(w) for w in W_cand]

        # Number of samples to use in this batch (considering max_samples limit)
        n_take = min(B, max_samples - collected)
        if n_take <= 0:
            break

        X_sel = x_1c[:n_take]  # [n_take, L]

        for sig in sigmas:
            # (Minimal modification) Reuse existing NumPy function for smoothing
            Y = np.stack([gaussian_smooth_1d(xi, sig) for xi in X_sel], axis=0)  # [n_take, L]
            Y_t = torch.from_numpy(Y).to(torch.float32)  # [B_sel, L]
            a_batch = dfa_alpha_batch_torch(Y_t, scales, order=1)  # [B_sel]

            # Collect valid values only
            a_np = a_batch.cpu().numpy()
            for a in a_np:
                if not np.isnan(a):
                    alphas.append(float(a))

        # Sample count increments based on "samples" same as original code
        collected += n_take
        if collected >= max_samples:
            break

    if len(alphas) == 0:
        # Fallback if data/scales are too short
        return [0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20][:max(0, num_levels-1)]

    alphas = np.array(alphas, dtype=float)
    # Quantile boundaries for equal distribution (1/num_levels, 2/num_levels, ..., (num_levels-1)/num_levels)
    qs = [(k / num_levels) for k in range(1, num_levels)]
    alpha_edges = np.quantile(alphas, qs).tolist()

    # Simple summary output (optional)
    print(f"[alpha edges] ({num_levels} levels) = {np.round(alpha_edges, 4).tolist()}")
    idx = np.searchsorted(alpha_edges, alphas, side="left")
    hist = [(idx == k).sum() for k in range(num_levels)]
    print(f"[bin counts] = {hist} (total={len(alphas)})")

    return alpha_edges

# Calculate alpha edges using equalized binning

In [3]:
alpha_edges = compute_alpha_edges_equalized(
    train_loader,
    max_samples=10000,
    num_levels=8,
    dfa_order=1,
    dfa_num_scales=12,
    dfa_smin_frac=0.02,
    dfa_smax_frac=0.25,
)

[alpha edges] (8 levels) = [1.553, 1.7421, 1.8511, 1.9343, 1.9968, 2.035, 2.0532]
[bin counts] = [np.int64(3000), np.int64(3000), np.int64(3000), np.int64(3000), np.int64(3000), np.int64(3000), np.int64(3000), np.int64(3000)] (total=24000)
